In [13]:
import pandas as pd

df=pd.read_csv("../spam.csv",encoding="latin-1")

print("Loaded")

Loaded


In [14]:
print(df.head(5))
print(df.columns)

     v1                                                 v2 Unnamed: 2  \
0   ham  Go until jurong point, crazy.. Available only ...        NaN   
1   ham                      Ok lar... Joking wif u oni...        NaN   
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...        NaN   
3   ham  U dun say so early hor... U c already then say...        NaN   
4   ham  Nah I don't think he goes to usf, he lives aro...        NaN   

  Unnamed: 3 Unnamed: 4  
0        NaN        NaN  
1        NaN        NaN  
2        NaN        NaN  
3        NaN        NaN  
4        NaN        NaN  
Index(['v1', 'v2', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'], dtype='str')


In [15]:
print(df.describe())
print(df.shape)
print(df.info())

          v1                      v2  \
count   5572                    5572   
unique     2                    5169   
top      ham  Sorry, I'll call later   
freq    4825                      30   

                                               Unnamed: 2  \
count                                                  50   
unique                                                 43   
top      bt not his girlfrnd... G o o d n i g h t . . .@"   
freq                                                    3   

                   Unnamed: 3 Unnamed: 4  
count                      12          6  
unique                     10          5  
top      MK17 92H. 450Ppw 16"    GNT:-)"  
freq                        2          2  
(5572, 5)
<class 'pandas.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   v1          5572 non-null   str  
 1   v2          5572 non-null   str  
 2   Unnamed: 2  

In [16]:
print(df.isnull().sum())

v1               0
v2               0
Unnamed: 2    5522
Unnamed: 3    5560
Unnamed: 4    5566
dtype: int64


In [17]:
print(df.duplicated().sum())

403


In [18]:
df = df.drop(["Unnamed: 2", "Unnamed: 3", "Unnamed: 4"], axis=1)
print(df.isnull().sum())

v1    0
v2    0
dtype: int64


In [19]:
df=df.drop_duplicates()
print(df.duplicated().sum())

0


In [20]:
import matplotlib.pyplot as plt

df["v1"].value_counts().plot(kind="bar")
plt.show()

In [23]:
df["message_length"] = df["v2"].str.len()

In [24]:
df["message_length"].hist(figsize=(8,5))
plt.xlabel("Message Length")
plt.ylabel("Frequency")
plt.show()

In [25]:
plt.figure(figsize=(10,10))
df.boxplot()
plt.show()

In [26]:
print(df.columns)

Index(['v1', 'v2', 'message_length'], dtype='str')


In [27]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer


tfidf=TfidfVectorizer()

le=LabelEncoder()
df["v1"]=le.fit_transform(df["v1"])
x=tfidf.fit_transform(df["v2"])
y=df["v1"]

x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)
print(x_train.shape)
print(y_train.shape)
print(x_test.shape)
print(y_test.shape)

(4135, 8672)
(4135,)
(1034, 8672)
(1034,)


In [28]:
print(x_train[:5])


<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 54 stored elements and shape (5, 8672)>
  Coords	Values
  (0, 5223)	0.21526418207941161
  (0, 8339)	0.3713098424423173
  (0, 7677)	0.4203347275451881
  (0, 3031)	0.5336641790294508
  (0, 4159)	0.5952407862869263
  (1, 7640)	0.4384404657527198
  (1, 859)	0.4867251638288413
  (1, 7669)	0.4082031492467239
  (1, 8355)	0.4206354282537766
  (1, 5531)	0.4767647200969724
  (2, 5504)	0.2607190359722852
  (2, 7640)	0.26835563401846035
  (2, 5334)	0.35269268893992584
  (2, 7721)	0.2757034291569376
  (2, 8209)	0.5742849137369284
  (2, 4698)	0.5742849137369284
  (3, 3358)	0.13466456136408833
  (3, 1084)	0.09315593646946087
  (3, 3308)	0.1013094785367915
  (3, 8433)	0.12074085356918199
  (3, 1813)	0.10879417192620001
  (3, 322)	0.22480814398761823
  (3, 5570)	0.1164168964507927
  (3, 1835)	0.1990069966434657
  (3, 298)	0.19075019913084018
  :	:
  (3, 8141)	0.1946115046917424
  (3, 5038)	0.1820548990131534
  (3, 4630)	0.18730719677859248
 

In [29]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

model=DecisionTreeClassifier()
model.fit(x_train,y_train)
y_pred=model.predict(x_test)

accuracy=accuracy_score(y_test,y_pred)
print("Accuracy:", accuracy)

Accuracy: 0.9671179883945842


In [30]:
from  sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

model = LogisticRegression(max_iter=100000)
model.fit(x_train, y_train)

y_pred = model.predict(x_test)
accuracy=accuracy_score(y_test,y_pred)
print("Accuracy:", accuracy)


Accuracy: 0.9555125725338491


In [31]:
from sklearn.metrics import confusion_matrix

cm=confusion_matrix(y_test,y_pred)
print(cm)

[[886   3]
 [ 43 102]]


In [32]:
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_classif

select = SelectKBest(score_func=f_classif, k=10)

x_new = select.fit_transform(x_train, y_train)

feature_names = tfidf.get_feature_names_out()

print("Selected Features:")
print(feature_names[select.get_support()])

Selected Features:
['150p' 'call' 'claim' 'free' 'mobile' 'prize' 'txt' 'uk' 'urgent' 'www']


C:\Users\tashw\AppData\Roaming\Python\Python313\site-packages\sklearn\feature_selection\_univariate_selection.py:110: UserWarning: Features [0 0 0 ... 0 0 0] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
C:\Users\tashw\AppData\Roaming\Python\Python313\site-packages\sklearn\feature_selection\_univariate_selection.py:111: RuntimeWarning: invalid value encountered in divide
  f = msb / msw


In [33]:
feature_names = tfidf.get_feature_names_out()

selected_features = feature_names[select.get_support()]

print(selected_features)

['150p' 'call' 'claim' 'free' 'mobile' 'prize' 'txt' 'uk' 'urgent' 'www']
